# 使い方（このページを開いたら、これだけ）

表のセルは**動画を作りません**。投稿もしません。
調査済みのアカウントから、「美容・ドッグフード・見守りカメラ・婚活を、どう作るか」を4枚出します。
最後のセルは、参考アカウントの型の秒数・カット・字幕のまま焼くジョブを書きます。mp4 は焼きません。

## 押すもの

1. 上のメニュー **ランタイム** → **すべてのセルを実行**
2. 「このノートブックは Google が作成したものではありません」と出たら **そのまま実行**
3. 終わるまで待つ。下に表と、ジャンルごとの文章が出る

コードは読まなくていい。日付の欄以外は触らなくていい。

## 出てきた表の見方

表は横に4ジャンル。数字は「伸びてる件数 / 伸びてない件数」。

- **冒頭** … 最初の3秒で、何から入るか
- **主役** … 画面の中心（人、手元、動物）
- **顔** … 顔を出すか
- **尺** … 何秒くらいか
- **音** … ナレーションか、自分の音か
- **強い** … 両方とも10件以上ある。比べてよい
- **弱い** … 件数が少ない。傾向として読む
- **比較不能** … 片方が3件未満。伸びてる側に多い型だけ使う
- **傾向** … 伸びてない側が薄いので、伸びてる側の多い型をそのまま書く

各ジャンルの文章は、上からこの順。

1. **オマージュ** … すず丸の型。ドッグフードと見守りだけ。集計には入っていない。映像やタイトルは真似しない
2. **このジャンルで作るとき** … 伸びてる側に多い作り
3. **やらないこと** … 伸びてない側に多いので、真似しない

ドッグフードを作るときは、表の「商品名から入る」より、オマージュの「同じ子の食いしん坊から入る」を先に使う。


In [ ]:
#@title 実行の設定（初めてならそのままでよい）
#@markdown 新しい調査CSVを置いたときだけ、フォルダ名（日付）を変える。今は 2026-10-03。
DATE = "2026-10-03"  #@param {type:"string"}
#@markdown 下は「何件あれば信じるか」。初めてなら変えない。
STRONG_MIN = 10  #@param {type:"integer"}
COMPARE_MIN = 3  #@param {type:"integer"}
THIN_BELOW = 10  #@param {type:"integer"}
MIN_GAP_PT = 10  #@param {type:"integer"}
BRANCH = "cursor/affi-template-bake-44d6"  #@param {type:"string"}

from pathlib import Path
import sys
import urllib.request

REPO = "fireworker011/Research"
LOCAL_ROOT = Path("research/affi-templates")
if not LOCAL_ROOT.is_dir():
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "research/affi-templates/affi_genre_templates.py").is_file():
            LOCAL_ROOT = candidate / "research/affi-templates"
            break
RAW = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/research/affi-templates"
DOWNLOAD = Path("/content/affi-templates") if Path("/content").is_dir() else Path(".affi-templates-download")

def _fetch(rel: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f"{RAW}/{rel}"
    print("取りました", rel)
    urllib.request.urlretrieve(url, dest)

local_py = LOCAL_ROOT / "affi_genre_templates.py"
local_csv = LOCAL_ROOT / "data" / DATE / "accounts.csv"
if local_py.is_file():
    sys.path.insert(0, str(LOCAL_ROOT))
else:
    _fetch("affi_genre_templates.py", DOWNLOAD / "affi_genre_templates.py")
    sys.path.insert(0, str(DOWNLOAD))

if local_csv.is_file():
    DATA_DIR = local_csv.parent
else:
    DATA_DIR = DOWNLOAD / "data" / DATE
    _fetch(f"data/{DATE}/accounts.csv", DATA_DIR / "accounts.csv")
    try:
        _fetch(f"data/{DATE}/videos.csv", DATA_DIR / "videos.csv")
    except Exception:
        print("動画一覧は無し。アカウント一覧だけで計算する。")
print("調査日", DATE, "を読みます。このあと下のセルで4ジャンルが出ます。")


In [ ]:
import affi_genre_templates as agt

try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown:
        def __init__(self, data):
            self.data = data
        def __repr__(self):
            return self.data
    def display(obj):
        print(getattr(obj, "data", obj))

built = agt.load_snapshot(
    DATA_DIR,
    agt.Thresholds(
        strong_min=STRONG_MIN,
        compare_min=COMPARE_MIN,
        thin_below=THIN_BELOW,
        min_gap_pt=MIN_GAP_PT,
    ),
)
out = LOCAL_ROOT / "templates" if local_csv.is_file() else DOWNLOAD / "templates"
agt.write_outputs(built, out)
display(Markdown(
    "## 読み方\n\n"
    "まず表。左がジャンル。冒頭は最初の3秒、主役は画面の中心。\n\n"
    "強い＝両方10件以上。弱い＝件数が少ない。比較不能＝片方が3件未満。\n\n"
    "ドッグフードと見守りは、表の次にある「オマージュ」を先に使う。"
))
display(Markdown(agt.render_compare(built)))
for item in built.genres:
    display(Markdown(agt.render_genre(item)))
print("ここまで。ファイルは", out, "にも書いてある。動画は作っていない。")


In [ ]:
#@title 焼くジョブを書く（mp4は焼かない）
#@markdown 4ジャンルそれぞれ、参考アカウントの型の秒数・カット・字幕のまま書く。表の尺の帯では秒を変えない。
from pathlib import Path

base = LOCAL_ROOT if (LOCAL_ROOT / "affi_genre_templates.py").is_file() else DOWNLOAD
needed = [
    "affi_bake.py",
    "affi_reference.py",
    "reference-accounts/looks.yaml",
    "reference-accounts/templates/the.care.logic.yaml",
    "reference-accounts/templates/nuts0629.yaml",
    "reference-accounts/templates/junjun_ranran.yaml",
    "reference-accounts/templates/yako.shiawasekon.yaml",
]
for rel in needed:
    dest = base / rel
    if not dest.is_file():
        _fetch(rel, dest)
if str(base) not in sys.path:
    sys.path.insert(0, str(base))
import affi_bake

out_root = Path("/content/affi-bake") if Path("/content").is_dir() else Path("affi-bake")
for item in built.genres:
    job = affi_bake.bake_genre(item.genre)
    affi_bake.annotate_table(
        job,
        duration_band=item.choice("duration_band").choice,
        opening_type=item.choice("opening_type").choice,
        main_subject=item.choice("main_subject").choice,
        face_shown=item.choice("face_shown").choice,
        basis=item.basis,
        account_confidence=item.account_confidence,
        homage=agt.homage_for(item.genre),
    )
    path = affi_bake.write_job(job, out_root / item.genre)
    print(item.genre, job["status"], "型の秒", job["duration_s"], path)
    for reason in job["blocked"]:
        print(" -", reason)
print("mp4 は焼いていない。投稿していない。")
